# Table 5 — Rule categorization breakdown per dataset

For each of the four article datasets, every mined rule is categorized as **Accept** / **Reject** / **Uncertain** based on whether its 95% CI lies entirely above, entirely below, or straddles the threshold (0.0).

We report this *twice*: once using the **uplift** CI (statistical significance) and once using the **realistic rule gain** CI (economic significance). The contrast is the central empirical claim of the article — statistical significance ≠ economic viability.

**Output:** `article/results/categorization_breakdown.csv` — consumed by `article/tables/table5_categorization.py`.
We use bootstrap percentile CIs throughout for consistency with the rest of the experimental evaluation.


In [1]:
from __future__ import annotations
import sys, warnings
from collections import Counter
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / 'notebooks' / 'article') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'notebooks' / 'article'))

warnings.filterwarnings('ignore')
from _datasets import load_all
from action_rules import ActionRules

OUT_CSV = REPO_ROOT / 'article' / 'results' / 'categorization_breakdown.csv'
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

In [2]:
N_BOOTSTRAP = 500
SEED = 42
THRESHOLD = 0.0
METRICS = ['uplift', 'realistic_rule_gain']

In [3]:
rows = []
for cfg in load_all():
    print(f'=== {cfg.name} ===')
    ar = ActionRules(
        min_stable_attributes=cfg.min_stable_attributes,
        min_flexible_attributes=cfg.min_flexible_attributes,
        min_undesired_support=cfg.min_undesired_support,
        min_desired_support=cfg.min_desired_support,
        min_undesired_confidence=cfg.min_undesired_confidence,
        min_desired_confidence=cfg.min_desired_confidence,
        intrinsic_utility_table=cfg.intrinsic_utility_table,
        transition_utility_table=cfg.transition_utility_table,
    )
    ar.fit(cfg.df, stable_attributes=cfg.stable_attributes, flexible_attributes=cfg.flexible_attributes,
           target=cfg.target, target_undesired_state=cfg.target_undesired_state,
           target_desired_state=cfg.target_desired_state, use_sparse_matrix=cfg.use_sparse_matrix)
    n_rules = len(ar.output.action_rules)
    print(f'  mined {n_rules} rules')
    for metric in METRICS:
        results = ar.confidence_intervals(
            cfg.df, method='bootstrap', confidence_level=0.95,
            threshold=THRESHOLD, metric=metric,
            n_bootstrap=N_BOOTSTRAP, random_state=SEED,
        )
        counts = Counter(r.category.value for r in results if r.category is not None)
        rows.append({
            'dataset': cfg.name,
            'metric': metric,
            'n_rules': n_rules,
            'accept': counts.get('accept', 0),
            'uncertain': counts.get('uncertain', 0),
            'reject': counts.get('reject', 0),
            'accept_pct': round(100 * counts.get('accept', 0) / n_rules, 1) if n_rules else 0.0,
            'reject_pct': round(100 * counts.get('reject', 0) / n_rules, 1) if n_rules else 0.0,
        })
        print(f'  {metric:>22}: A={counts.get("accept",0)}, U={counts.get("uncertain",0)}, R={counts.get("reject",0)}')
df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)
print(f'\nwrote {OUT_CSV.relative_to(REPO_ROOT)}')
df

=== Telco Customer Churn ===


  mined 24 rules


                  uplift: A=24, U=0, R=0


     realistic_rule_gain: A=24, U=0, R=0
=== UCI Bank Marketing ===


  mined 24 rules


...

     realistic_rule_gain: A=12, U=1, R=8
=== Taiwan Credit Card Default ===


  mined 20 rules


                  uplift: A=20, U=0, R=0


     realistic_rule_gain: A=20, U=0, R=0

wrote article\results\categorization_breakdown.csv


,dataset,metric,n_rules,accept,uncertain,reject,accept_pct,reject_pct
0,Telco Customer Churn,uplift,24,24,0,0,100.0,0.0
1,Telco Customer Churn,realistic_rule_gain,24,24,0,0,100.0,0.0
2,UCI Bank Marketing,uplift,24,24,0,0,100.0,0.0
3,UCI Bank Marketing,realistic_rule_gain,24,24,0,0,100.0,0.0
4,IBM Employee Attrition,uplift,21,20,1,0,95.2,0.0
5,IBM Employee Attrition,realistic_rule_gain,21,12,1,8,57.1,38.1
6,Taiwan Credit Card Default,uplift,20,20,0,0,100.0,0.0
7,Taiwan Credit Card Default,realistic_rule_gain,20,20,0,0,100.0,0.0


## Interpretation

Compare `accept_pct` for **uplift** vs **realistic_rule_gain** for each dataset. Where the gain breakdown shows a non-trivial *reject* count while the uplift breakdown is mostly *accept*, the dataset exhibits the **statistical vs economic significance gap**: rules that pass a statistical test would lose money in deployment because the intervention cost outweighs the expected outcome change.

The IBM Employee Attrition dataset is the cleanest demonstration of this gap in the article (rules involving large salary increases tend to have negative realistic gain).
